# 1.7) Object-oriented programming for natural systems

Some data naturally travels with the operations that act on it: a weather station has a name, an elevation, and a growing list of readings, and the things you do with it — add a reading, compute its mean — belong to that station. A class bundles such state and behaviour into one object. This subchapter builds a `WeatherStation`, composes stations into a network, contrasts a lightweight `dataclass` record, and is careful about the opposite lesson too: when a plain function is the better tool. The generated-code bug at the end is the object-oriented twin of the shared-default-list trap from earlier.

:::{admonition} Learning objectives
:class: tip
- Define a class with __init__ and self, and create instances.
- Distinguish instance attributes (unique per object) from class attributes (shared).
- Write type-hinted methods that read state and methods that mutate it, and a useful __repr__.
- Use a dataclass for a lightweight record, and composition to build larger objects from smaller ones.
- Judge when a class earns its keep and when a function is clearer.
:::

## Classes and instances

A class is a blueprint; an instance is one object built from it. `__init__` runs at construction and stores the object's data on `self`, the reference to the instance being built.

In [1]:
class WeatherStation:
    # class attribute: one value shared by every station (a physical constant)
    lapse_rate_celsius_per_km = -6.5

    def __init__(self, name: str, elevation_m: float):
        self.name = name                          # instance attributes: per-object
        self.elevation_m = elevation_m
        self.readings_celsius: list[float] = []   # each station owns its own list

    def add_reading(self, temp_celsius: float) -> None:
        # a method that MUTATES state
        self.readings_celsius.append(temp_celsius)

    def mean_temperature(self) -> float:
        # a method that READS state
        if not self.readings_celsius:
            raise ValueError(f"{self.name} has no readings")
        return sum(self.readings_celsius) / len(self.readings_celsius)

    def sea_level_temperature(self, temp_celsius: float) -> float:
        # uses the shared class attribute together with instance state
        return temp_celsius - self.lapse_rate_celsius_per_km * (self.elevation_m / 1000)

    def __repr__(self) -> str:
        return f"WeatherStation({self.name!r}, {self.elevation_m} m, n={len(self.readings_celsius)})"

In [2]:
jungfraujoch = WeatherStation("Jungfraujoch", 3571)
basel = WeatherStation("Basel", 316)

for t in [-2.3, -1.1, 0.4]:
    jungfraujoch.add_reading(t)
for t in [18.0, 19.2, 17.5]:
    basel.add_reading(t)

print(jungfraujoch)                                  # __repr__ in action
print(basel)
print("JFJ mean:", round(jungfraujoch.mean_temperature(), 2), "°C")
print("JFJ at sea level:", round(jungfraujoch.sea_level_temperature(0.0), 1), "°C")

WeatherStation('Jungfraujoch', 3571 m, n=3)
WeatherStation('Basel', 316 m, n=3)
JFJ mean: -1.0 °C
JFJ at sea level: 23.2 °C


## Instance versus class attributes

An *instance* attribute belongs to one object (each station's own `readings_celsius`). A *class* attribute is shared by every instance (the single `lapse_rate_celsius_per_km`). Shared **constants** are a good use of class attributes; shared **mutable** state is the trap shown later.

In [3]:
# the lapse rate is one shared value, reachable via the class or any instance
print(WeatherStation.lapse_rate_celsius_per_km,
      jungfraujoch.lapse_rate_celsius_per_km,
      basel.lapse_rate_celsius_per_km)

# the readings are independent per instance
print("JFJ readings:", jungfraujoch.readings_celsius)
print("Basel readings:", basel.readings_celsius)

-6.5 -6.5 -6.5
JFJ readings: [-2.3, -1.1, 0.4]
Basel readings: [18.0, 19.2, 17.5]


## dataclasses: records with less ceremony

When an object is mostly a bundle of fields, `@dataclass` generates `__init__`, `__repr__`, and `__eq__` for you from type-annotated attributes.

In [4]:
from dataclasses import dataclass

@dataclass
class Reading:
    timestamp: str
    temp_celsius: float

r = Reading("2024-06-01", 18.2)
print(r)                                                  # auto __repr__
print(r.temp_celsius)
print(Reading("2024-06-01", 18.2) == Reading("2024-06-01", 18.2))   # auto __eq__

Reading(timestamp='2024-06-01', temp_celsius=18.2)
18.2
True


## Composition: build larger objects from smaller ones

Composition is a *has-a* relationship: a `StationNetwork` *has* stations. The container delegates work to the objects it holds rather than re-implementing it.

In [5]:
class StationNetwork:
    def __init__(self, name: str):
        self.name = name
        self.stations: dict[str, WeatherStation] = {}

    def add_station(self, station: WeatherStation) -> None:
        self.stations[station.name] = station

    def coldest_station(self) -> WeatherStation:
        # delegate to each station's own mean_temperature method
        return min(self.stations.values(), key=lambda s: s.mean_temperature())

    def __repr__(self) -> str:
        return f"StationNetwork({self.name!r}, {len(self.stations)} stations)"

network = StationNetwork("Switzerland")
network.add_station(jungfraujoch)
network.add_station(basel)
print(network)
print("coldest:", network.coldest_station().name)

StationNetwork('Switzerland', 2 stations)
coldest: Jungfraujoch


:::{admonition} Computational-thinking fundamental: reach for a class only when there is state to bundle
:class: important
A class earns its keep when data and the operations on it belong together and the object carries state between calls — a station accumulating readings, a model remembering what it learned. When there is no state to keep, an object is just ceremony around a function. Model entities as objects; model transformations as functions. Knowing which is which keeps a codebase small and honest.
:::

## When not to use a class

A stateless transformation needs no object. Wrapping a one-line conversion in a class adds boilerplate and hides a simple function behind a constructor.

In [6]:
# a pure function is the right tool here: input in, output out, no state
def celsius_to_kelvin(temp_celsius: float) -> float:
    return temp_celsius + 273.15

print(celsius_to_kelvin(18.2))
# a class with no attributes and a single method would only add ceremony

291.34999999999997


## When generated code lies: the shared class-level list

Asked for a station class, an assistant declares the readings list at the class level. That single list is then shared by every instance — the object-oriented version of the mutable-default-argument bug.

In [7]:
class WeatherStationBuggy:
    readings_celsius = []        # class attribute: ONE list shared by all instances

    def __init__(self, name):
        self.name = name

    def add_reading(self, temp_celsius):
        self.readings_celsius.append(temp_celsius)

a = WeatherStationBuggy("A")
b = WeatherStationBuggy("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.readings_celsius)   # expected [10.0]
print("B readings:", b.readings_celsius)   # expected [20.0]

A readings: [10.0, 20.0]
B readings: [10.0, 20.0]


:::{admonition} Diagnosis: mutable state declared on the class is shared
:class: warning
`readings_celsius = []` sits on the class, so there is exactly one list and every instance appends to it; station A and station B end up with each other's data, silently. Mutable per-instance state must be created inside `__init__` with `self.readings_celsius = []`, so each object gets its own list. Class attributes are for values that are genuinely shared and, ideally, immutable.
:::

In [8]:
class WeatherStationFixed:
    def __init__(self, name):
        self.name = name
        self.readings_celsius = []      # per-instance list, created on construction

    def add_reading(self, temp_celsius):
        self.readings_celsius.append(temp_celsius)

a = WeatherStationFixed("A")
b = WeatherStationFixed("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.readings_celsius, "| B readings:", b.readings_celsius)

A readings: [10.0] | B readings: [20.0]


:::{admonition} Going deeper: inheritance
:class: seealso dropdown
A subclass inherits a parent's attributes and methods, calling `super().__init__` to reuse the parent's constructor, then adding its own.

```python
class RiverGauge(WeatherStation):
    def __init__(self, name, elevation_m):
        super().__init__(name, elevation_m)
        self.discharge_m3s = []          # extra state specific to a gauge

    def add_discharge(self, q_m3s):
        self.discharge_m3s.append(q_m3s)
```

Prefer composition to deep inheritance hierarchies; inherit only for a genuine *is-a* relationship.
:::

:::{admonition} Going deeper: property validation
:class: seealso dropdown
A `@property` lets an attribute run validation on assignment while still being accessed like plain data.

```python
class Station:
    def __init__(self, elevation_m):
        self.elevation_m = elevation_m       # goes through the setter

    @property
    def elevation_m(self):
        return self._elevation_m

    @elevation_m.setter
    def elevation_m(self, value):
        if value < -500:
            raise ValueError("elevation below -500 m is implausible")
        self._elevation_m = value
```
:::

:::{admonition} Going deeper: abstract base classes
:class: seealso dropdown
An abstract base class defines an interface that subclasses must implement, enforced at instantiation.

```python
from abc import ABC, abstractmethod

class Sensor(ABC):
    @abstractmethod
    def read(self) -> float:
        ...
# a subclass that does not implement read() cannot be instantiated
```

ABCs document the contract a family of objects must satisfy.
:::

:::{admonition} Going deeper: the scikit-learn estimator-as-object pattern
:class: seealso dropdown
scikit-learn models are objects that learn state in `fit` (stored on attributes with a trailing underscore) and use it in `predict` — the bridge to the machine-learning subchapter.

```python
class MeanRegressor:
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)     # learned state
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]
```

Every estimator you will meet follows this fit/predict object shape.
:::

:::{admonition} Takeaways
:class: danger
- A class bundles state (attributes) and behaviour (methods); `__init__` stores data on `self` at construction.
- Instance attributes are per-object; class attributes are shared — good for constants, dangerous for mutable state.
- Write a `__repr__` for readable objects; type-hint methods; separate methods that read state from those that mutate it.
- Use a `dataclass` for plain records, and composition to assemble larger objects from smaller ones.
- A stateless transformation should be a function, not a class.
- Never put mutable state (`readings = []`) on the class body: it is shared across all instances. Initialise it in `__init__`.
:::

## Resources

- [Object-Oriented Programming (OOP) in Python](https://realpython.com/python3-object-oriented-programming/) — classes, instances, attributes, methods, and inheritance, with worked examples.
- [Python Classes: The Power of Object-Oriented Programming](https://realpython.com/python-classes/) — instance vs class attributes, dataclasses, abstract base classes, and explicit guidance on when *not* to use a class.
:::